In [ ]:
#model-1434439362270986240_tflite_2025-12-03T11_31_39.916325Z_model.tflite

In [2]:
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image
import json

class AutoMLEdgeLocalPredictor:
    def __init__(self, model_path: str):
        """Load the exported AutoML Edge .tflite model"""
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        
        # Get input/output details
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        
        # AutoML Edge typically expects 224x224x3 RGB input
        self.input_shape = self.input_details[0]['shape']
        print(f"Model input shape: {self.input_shape}")
    
    def preprocess_image(self, image_path: str) -> np.ndarray:
        """Preprocess image for AutoML Edge model (224x224x3, normalized)"""
        # Load and resize image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Resize to model input size (typically 224x224)
        h, w = self.input_shape[1], self.input_shape[2]
        image = cv2.resize(image, (w, h))
        
        # Convert to float32 and normalize [0,1]
        image = image.astype(np.float32) / 255.0
        
        # Add batch dimension
        image = np.expand_dims(image, axis=0)
        
        return image
    
    def predict(self, image_path: str, top_k: int = 3) -> dict:
        """Run prediction on local image"""
        # Preprocess
        input_data = self.preprocess_image(image_path)
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        
        # Run inference
        self.interpreter.invoke()
        
        # Get predictions
        output_data = self.interpreter.get_tensor(self.output_details[0]['index'])
        predictions = output_data[0]  # Remove batch dimension
        
        # Get top-k predictions with scores
        top_indices = np.argsort(predictions)[::-1][:top_k]
        results = []
        
        for i in top_indices:
            confidence = float(predictions[i])
            results.append({
                "label": f"class_{i}",  # Replace with your actual class names
                "confidence": confidence
            })
        
        return {"predictions": results}

# Your label mapping (replace with your actual class names from training)
LABEL_MAP = {
    0: "pan_card",
    1: "aadhar_front", 
    2: "aadhar_back",
    3: "aadhar_full"
}


# Usage example
def main():
    # Initialize predictor with your exported .tflite model
    modelpath = './models/small-data-trained-models/model-1434439362270986240_tflite_2025-12-03T11_31_39.916325Z_model.tflite'
    predictor = AutoMLEdgeLocalPredictor(modelpath)  # Path to your .tflite file
    
    # Predict on KYC document
    image_path = './test-data/Aadhar_715c658c-6de9-40ef-a18f-7d17cbf5dbdb.jpg'  # Your test image path
    
    results = predictor.predict(image_path, top_k=3)
    
    # Map indices to actual labels
    for pred in results["predictions"]:
        pred["label"] = LABEL_MAP.get(int(pred["label"].split("_")[1]), pred["label"])
        print(f"Label: {pred['label']}, Confidence: {pred['confidence']:.3f}")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'tensorflow'